# 01a Environment Variables (Hugging Face)

This notebook validates Hugging Face access without using the CLI. It relies on the `HF_TOKEN` environment variable to authenticate and then performs a small, controlled download check.

Run order:
1. Set `HF_TOKEN` (and optional test model env vars).
2. Validate token + account access.
3. Run the model/tokenizer download test.


## 1) Set `HF_TOKEN` (no CLI)

You must create a Hugging Face access token and set it as an environment variable **before** running the auth check.

### Option A: Set in your shell

```bash
export HF_TOKEN="your_hf_token_here"
```

### Option B: Set inside the notebook (temporary)

Use the next cell to paste your token (input is hidden). This sets `HF_TOKEN` for the current kernel only.

Notes:
- Make sure you have accepted any gated model licenses you plan to download.
- Do **not** print or share your token.


In [1]:
import os  # Python standard library (environment variables)
from getpass import getpass  # Python standard library (hidden input)


def get_hf_token() -> str:
    """Return a usable Hugging Face token from env or a hidden prompt.

    Raises:
        ValueError: If the token is missing or empty after prompting.
    """
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    print("HF_TOKEN not found in environment.")
    token = getpass("Paste HF_TOKEN (input hidden): ").strip()
    if not token:
        raise ValueError("HF_TOKEN is required to authenticate with Hugging Face.")

    # Set it for this kernel session only (not persisted to your shell).
    os.environ["HF_TOKEN"] = token
    return token


hf_token = get_hf_token()
print("HF_TOKEN loaded (masked):", f"{hf_token[:4]}...{hf_token[-4:]}")

HF_TOKEN not found in environment.
Paste HF_TOKEN (input hidden): ··········
HF_TOKEN loaded (masked): hf_T...PFTE


In [2]:
from huggingface_hub import whoami  # huggingface_hub package (HF auth utilities)

try:
    user_info = whoami(token=hf_token)
    print("Hugging Face authentication: ✓")
    print(f"Logged in as: {user_info.get('name', 'Unknown')}")
except Exception as exc:
    print("Hugging Face authentication: ✗")
    print(f"Error: {exc}")
    raise

Hugging Face authentication: ✓
Logged in as: tmeisenzahl


## 2) Configure the download test

To avoid hardcoding any model IDs, set a small, public model ID in your environment.

### Option A: Set in your shell

```bash
export HF_TEST_MODEL_ID="<your-small-public-model-id>"
```

If you use the shell approach, make sure you **launch Jupyter from the same shell** so the kernel inherits the environment variables.

### Option B: Set inside the notebook (temporary)

Use the next cell to set `HF_TEST_MODEL_ID` for this kernel session only.

Optional:
- Set `HF_DOWNLOAD_WEIGHTS=true` if you want to download full model weights.
- If omitted, the test only downloads the model config + tokenizer.


# Look at the top of your VS Code and copy paste google/gemma-3-1b-it into that command!!!

google/gemma-3-1b-it

In [3]:
import os  # Python standard library (environment variables)


def get_test_model_id() -> str:
    """Return a test model ID from env or prompt the user.

    Raises:
        ValueError: If the model ID is missing or empty after prompting.
    """
    model_id = os.environ.get("HF_TEST_MODEL_ID")
    if model_id:
        return model_id

    print("HF_TEST_MODEL_ID not found in environment.")
    model_id = input("Enter a small public model ID: ").strip()
    if not model_id:
        raise ValueError("HF_TEST_MODEL_ID is required for the download test.")

    # Set it for this kernel session only (not persisted to your shell).
    os.environ["HF_TEST_MODEL_ID"] = model_id
    return model_id


test_model_id = get_test_model_id()
print("HF_TEST_MODEL_ID set for this kernel:", test_model_id)

HF_TEST_MODEL_ID not found in environment.
Enter a small public model ID: google/gemma-3-1b-it
HF_TEST_MODEL_ID set for this kernel: google/gemma-3-1b-it


In [4]:
from transformers import AutoConfig, AutoModel, AutoTokenizer  # transformers package (model utilities)

model_id = globals().get("test_model_id") or os.environ.get("HF_TEST_MODEL_ID")
if not model_id:
    raise ValueError(
        "HF_TEST_MODEL_ID is not set. Set it to a small public model ID and rerun."
    )

download_weights = os.environ.get("HF_DOWNLOAD_WEIGHTS", "true").lower() == "true"

print(f"Download test model_id: {model_id}")
print(f"Download full weights: {download_weights}")

try:
    config = AutoConfig.from_pretrained(model_id, token=hf_token)
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    print("Config + tokenizer download: ✓")

    if download_weights:
        _ = AutoModel.from_pretrained(model_id, token=hf_token)
        print("Model weights download: ✓")
    else:
        print("Model weights download: skipped (set HF_DOWNLOAD_WEIGHTS=true to enable)")
except Exception as exc:
    print("Download test failed.")
    print(f"Error: {exc}")
    raise

Download test model_id: google/gemma-3-1b-it
Download full weights: True


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Config + tokenizer download: ✓


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model weights download: ✓


# Dataset Loading and Exploration

This notebook loads semantic similarity datasets for fine-tuning EmbeddingGemma. We'll explore the toy dataset and prepare it for training.


```
# Import data loading functions from the scripts directory
# This reinforces the rule that ALL functions must live in /src scripts
from src.data.loaders import load_toy_dataset, validate_pairs
import pandas as pd
```

## Load Toy Dataset
The toy dataset contains 4 pairs of semantically similar sentences. This small dataset is perfect for demonstrating the fine-tuning process.

```
# Using the imported function to load the toy dataset
# No logic is embedded in the notebook – notebooks are for workflow execution only
train_data = load_toy_dataset()

# Display the dataset in a table format for exploration
df = pd.DataFrame(train_data)
df
```
## Dataset Statistics
Let's examine the characteristics of our dataset.

```
# Validate and get statistics about the dataset
stats = validate_pairs(train_data)

print("Dataset Statistics:")
print("-" * 40)
print(f"Number of pairs: {stats['count']}")
print(f"Average anchor length: {stats['avg_anchor_length']:.1f} characters")
print(f"Average positive length: {stats['avg_positive_length']:.1f} characters")
print(f"Contains empty strings: {stats['has_empty']}")
```
## Dataset Overview

Each pair consists of an anchor sentence and a positive (similar) sentence. The model will learn to make embeddings of these pairs close together in vector space.

```
# Display each pair for inspection
for i, pair in enumerate(train_data, 1):
    print(f"\nPair {i}:")
    print(f"  Anchor:   '{pair['anchor']}'")
    print(f"  Positive: '{pair['positive']}'")
```
